## 1.0 Set up File

1.1 Import packages

In [6]:
import pandas as pd
from PIL import Image
import numpy as np
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf  # For tf.data
import matplotlib.pyplot as plt
import keras
from keras import layers
from keras.applications import EfficientNetB0

1.2 Define Model Parameters

In [7]:
IMG_SIZE = 224
BATCH_SIZE = 64

1.3 Read in Data

In [8]:
# Load dataframe
df = pd.read_csv("fossils_augmented.csv")

# Shuffle the dataframe for tensorflow training
df = df.sample(frac=1, random_state=10).reset_index(drop=True)

In [10]:
print(f"Total images: {len(df)}")
print(df["period"].value_counts())

Total images: 3611
period
Silurian Period         1024
Ordovician Period        337
Permian Period           250
Carboniferous Period     250
Jurassic Period          250
Cretaceous Period        250
Triassic Period          250
Cambrian Period          250
Precambrian Eon          250
Devonian Period          250
Cenozoic Era             250
Name: count, dtype: int64


1.4 Train/Test Split Data

In [11]:
# Split so no augmented images are in validation/test sets
train_df = df[df["augmented"] == True]
orig_df  = df[df["augmented"] == False]

# 80/20 split on originals only
val_df   = orig_df.sample(frac=0.2, random_state=42)
train_orig_df = orig_df.drop(val_df.index)

# Combine original training rows with augmented rows
train_df = pd.concat([train_orig_df, train_df], ignore_index=True).sample(frac=1, random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Train: 3144 | Val: 467


1.5 Run DataGenerators on Images

In [13]:
train_datagen = ImageDataGenerator()
val_datagen   = ImageDataGenerator()

train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col="local_path",
    y_col="period",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=1
)

val_gen = val_datagen.flow_from_dataframe(
    val_df,
    x_col="local_path",
    y_col="period",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 3144 validated image filenames belonging to 11 classes.
Found 467 validated image filenames belonging to 11 classes.


1.6 Save Class Names and Batches

In [14]:
# Save class names for prediction later
import json
class_names = list(train_gen.class_indices.keys())
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print(f"\nClasses: {class_names}")
print(f"Training batches:   {len(train_gen)}")
print(f"Validation batches: {len(val_gen)}")


Classes: ['Cambrian Period', 'Carboniferous Period', 'Cenozoic Era', 'Cretaceous Period', 'Devonian Period', 'Jurassic Period', 'Ordovician Period', 'Permian Period', 'Precambrian Eon', 'Silurian Period', 'Triassic Period']
Training batches:   50
Validation batches: 8


## 2.0 Create Model